In [2]:
import torch

In [3]:
def conv2d_scratch(image, kernel):
    # image: tensor shape (h, w)
    # kernel: tensor shape (h, w)

    h, w = image.shape
    kh, kw = kernel.shape

    # calculate output shape
    out_h = h - kh + 1
    out_w = w - kw + 1

    # initialize output tensor
    output = torch.zeros(out_h, out_w)

    # perform convolution
    for i in range(out_h):
        for j in range(out_w):
            # split image into patches
            region = image[i:i + kh, j:j + kw]

            # dot product of region and kernel
            dot = region * kernel
            output[i, j] = dot.sum()

    return output


In [4]:
image = torch.tensor([
    [0, 0, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0]
], dtype=torch.float32)

# Filter detect đường thẳng đứng
kernel = torch.tensor([
    [0, 1, 0],
    [0, 1, 0],
    [0, 1, 0]
], dtype=torch.float32)

output = conv2d_scratch(image, kernel)
print(output)

tensor([[1., 3., 1.],
        [1., 3., 1.],
        [0., 2., 0.]])


In [5]:
import torch.nn.functional as F

#pytorch conv2d (batch, channel, height, width)
image_pt = image.unsqueeze(0).unsqueeze(0)  # (1, 1, 5, 5)
kernel_pt = kernel.unsqueeze(0).unsqueeze(0)  # (1, 1, 3, 3)
output_pt = F.conv2d(image_pt, kernel_pt)
print(output_pt.squeeze())

tensor([[1., 3., 1.],
        [1., 3., 1.],
        [0., 2., 0.]])


In [6]:
print(kernel_pt)
print(image_pt)

tensor([[[[0., 1., 0.],
          [0., 1., 0.],
          [0., 1., 0.]]]])
tensor([[[[0., 0., 1., 0., 0.],
          [0., 1., 1., 1., 0.],
          [0., 0., 1., 0., 0.],
          [0., 0., 1., 0., 0.],
          [0., 0., 0., 0., 0.]]]])


In [7]:
def conv2d_multi_channel(image, kernels):
    # image shape: (c_in, h, w)
    # kernels shape: (c_out, c_in, h, w) - n filters, each filter has c_in channels

    c_in, h, w = image.shape
    c_out, _, kh, kw = kernels.shape

    out_h = h - kh + 1
    out_w = w - kw + 1
    output = torch.zeros(c_out, out_h, out_w)

    # for each filter
    for f in range(c_out):
        # for each channel
        for c in range(c_in):
            # for each pixel
            for i in range(out_h):
                for j in range(out_w):
                    region = image[c, i:i+kh, j:j+kw]
                    output[f,i,j] += (region * kernels[f, c]).sum()

    return output


In [8]:
torch.manual_seed(42)
image_rgb = torch.randint(0, 2, (3, 5, 5)).float()

# 2 random filters - each filter has 3 channels 3x3x3
kernels = torch.randint(0, 2, (2, 3, 3, 3)).float()

output = conv2d_multi_channel(image_rgb, kernels)
print("Output shape:", output.shape)  # phải là (2, 3, 3)
print(output)

Output shape: torch.Size([2, 3, 3])
tensor([[[ 7.,  8.,  5.],
         [ 7.,  7.,  7.],
         [ 8.,  9., 10.]],

        [[ 5.,  7.,  5.],
         [ 6.,  8.,  8.],
         [ 8.,  9.,  8.]]])


In [9]:
image_pt = image_rgb.unsqueeze(0)    # (1, 3, 5, 5)
kernels_pt = kernels                  # (2, 3, 3, 3)

output_pt = F.conv2d(image_pt, kernels_pt)
print(output_pt.squeeze())

tensor([[[ 7.,  8.,  5.],
         [ 7.,  7.,  7.],
         [ 8.,  9., 10.]],

        [[ 5.,  7.,  5.],
         [ 6.,  8.,  8.],
         [ 8.,  9.,  8.]]])


In [10]:
import torch.nn as nn

class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()

        # block 1: conv -> reLU -> maxPool
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        # block 2: conv -> reLU -> maxPool
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        # fully connected layers
        self.fc1 = nn.Linear(16*4*4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        # block 1
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)

        # block 2
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool2(x)

        # flatten before feeding into fully connected layers
        x = x.view(x.size(0), -1) # (batch, 16*4*4) = (batch, 256)

        # fully connected
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

model = LeNet()
dummy_input = torch.randn(1, 1, 28, 28)  # 1 ảnh, 1 channel, 28x28
output = model(dummy_input)
print("Output shape:", output.shape)  # phải là (1, 10)

Output shape: torch.Size([1, 10])


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # mean và std của MNIST
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Khởi tạo model, loss, optimizer
model = LeNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train loop
def train(model, loader, criterion, optimizer, epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(loader):
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Tracking
        total_loss = total_loss + loss.item()
        predicted = outputs.argmax(dim=1)
        correct = correct + (predicted == labels).sum().item()
        total = total + labels.size(0)

    accuracy = correct / total * 100
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch} — Loss: {avg_loss:.4f} — Accuracy: {accuracy:.2f}%")

# Test loop
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            predicted = outputs.argmax(dim=1)
            correct = correct + (predicted == labels).sum().item()
            total = total + labels.size(0)

    accuracy = correct / total * 100
    print(f"Test Accuracy: {accuracy:.2f}%")

# Chạy 5 epochs
for epoch in range(1, 6):
    train(model, train_loader, criterion, optimizer, epoch)

evaluate(model, test_loader)

Epoch 1 — Loss: 0.2310 — Accuracy: 92.99%
Epoch 2 — Loss: 0.0668 — Accuracy: 97.90%
Epoch 3 — Loss: 0.0489 — Accuracy: 98.52%
Epoch 4 — Loss: 0.0379 — Accuracy: 98.82%
Epoch 5 — Loss: 0.0328 — Accuracy: 98.96%
Test Accuracy: 98.58%


In [12]:
import torch.nn as nn

class LeNetBN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(LeNetBN, self).__init__()

        # Block 1: Conv → BN → ReLU → MaxPool
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        self.bn1 = nn.BatchNorm2d(num_features=6)
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        # block 2: Conv → BN → ReLU → MaxPool
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.bn2 = nn.BatchNorm2d(num_features=16)
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        # Fully connected + Dropout
        self.fc1 = nn.Linear(16*4*4, 120)
        self.bn3 = nn.BatchNorm1d(num_features=120)  # 1D after flatten
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        # block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool1(x)

        # block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.pool2(x)

        # Flatten
        x = x.view(x.size(0), -1) # (batch, 16*4*4) = (batch, 256)

        # FC layers với BN và Dropout
        x = self.fc1(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.relu(x)

        x = self.fc3(x)

        return x

model = LeNet()
dummy_input = torch.randn(1, 1, 28, 28)  # 1 ảnh, 1 channel, 28x28
output = model(dummy_input)
print("Output shape:", output.shape)  # phải là (1, 10)

Output shape: torch.Size([1, 10])


In [13]:
def train_model(model, train_loader, test_loader, num_epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(1, num_epochs + 1):
        # Train
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
            predicted = outputs.argmax(dim=1)
            correct = correct + (predicted == labels).sum().item()
            total = total + labels.size(0)

        train_acc = correct / total * 100
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch} — Loss: {avg_loss:.4f} — Train Acc: {train_acc:.2f}%")

    # Evaluate
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            predicted = outputs.argmax(dim=1)
            correct = correct + (predicted == labels).sum().item()
            total = total + labels.size(0)

    test_acc = correct / total * 100
    print(f"→ Test Accuracy: {test_acc:.2f}%\n")
    return test_acc

# Train LeNet gốc
print("=== LeNet gốc ===")
model_original = LeNet()
acc_original = train_model(model_original, train_loader, test_loader)

# Train LeNet + BN + Dropout
print("=== LeNet + BN + Dropout ===")
model_bn = LeNetBN(dropout_rate=0.5)
acc_bn = train_model(model_bn, train_loader, test_loader)

# So sánh
print("=" * 35)
print(f"LeNet gốc         : {acc_original:.2f}%")
print(f"LeNet + BN+Dropout: {acc_bn:.2f}%")

=== LeNet gốc ===
Epoch 1 — Loss: 0.2446 — Train Acc: 92.48%
Epoch 2 — Loss: 0.0705 — Train Acc: 97.81%
Epoch 3 — Loss: 0.0495 — Train Acc: 98.39%
Epoch 4 — Loss: 0.0403 — Train Acc: 98.76%
Epoch 5 — Loss: 0.0346 — Train Acc: 98.88%
→ Test Accuracy: 98.42%

=== LeNet + BN + Dropout ===
Epoch 1 — Loss: 0.2316 — Train Acc: 93.62%
Epoch 2 — Loss: 0.0883 — Train Acc: 97.28%
Epoch 3 — Loss: 0.0721 — Train Acc: 97.82%
Epoch 4 — Loss: 0.0622 — Train Acc: 98.19%
Epoch 5 — Loss: 0.0553 — Train Acc: 98.31%
→ Test Accuracy: 99.01%

LeNet gốc         : 98.42%
LeNet + BN+Dropout: 99.01%
